In [1]:
from pyavro import namevalidator

ModuleNotFoundError: No module named 'pyavro'

In [24]:
class A:
    def __hash__(self):
        return 1
    def __eq__(self, value):
        return True

a1 = A()
a2 = A()
d = {}
d[a1] = a1
print(a2 in d)
print(a1 in d)

True
True


In [25]:
class IdentityDict(dict):
    def __setitem__(self, key, value):
        return super().__setitem__(id(key), value)
    def __getitem__(self, key):
        return super().__getitem__(id(key))
    def __contains__(self, key):
        return super().__contains__(id(key))

In [26]:
idict = IdentityDict()

In [27]:
idict[a1] = 'a1'

In [28]:
idict[a1]

'a1'

In [29]:
a2 in idict

False

In [33]:
id(None)

140723622892112

In [ ]:

from collections.abc import MutableMapping
from typing import Iterator, ItemsView, KeysView, ValuesView, Tuple, Any

class IdentityDict(MutableMapping):
    """
    Identity-based mapping: keys are compared by object identity (is), not equality (==).
    Strong references are kept to keys to avoid id reuse issues.
    """
    def __init__(self) -> None:
        self._data: dict[int, tuple[object, Any]] = {}

    def __setitem__(self, key: object, value: Any) -> None:
        self._data[id(key)] = (key, value)

    def __getitem__(self, key: object) -> Any:
        return self._data[id(key)][1]

    def __delitem__(self, key: object) -> None:
        del self._data[id(key)]

    def __contains__(self, key: object) -> bool:  # identity semantics
        return id(key) in self._data

    def __iter__(self) -> Iterator[object]:
        # iterate original keys
        for k, _ in self._data.values():
            yield k

    def __len__(self) -> int:
        return len(self._data)

    # Helpful dict-like APIs
    def get(self, key: object, default: Any = None) -> Any:
        entry = self._data.get(id(key))
        return entry[1] if entry is not None else default

    def pop(self, key: object, default: Any = None) -> Any:
        iid = id(key)
        if iid in self._data:
            _, value = self._data.pop(iid)
            return value
        if default is not None:
            return default
        raise KeyError(key)

    def clear(self) -> None:
        self._data.clear()

    def items(self) -> Iterator[Tuple[object, Any]]:
        for k, v in self._data.values():
            yield (k, v)

    def keys(self) -> Iterator[object]:
        for k, _ in self._data.values():
            yield k

    def values(self) -> Iterator[Any]:
        for _, v in self._data.values():
            yield v

    # Optional: equality semantics similar to IdentityHashMap (compare entries by key identity)
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, IdentityDict):
            return NotImplemented
        # Compare by identity of keys and equality of values
        if len(self) != len(other):
            return False
        # Build a map of id(key) -> value for the other dict
        other_map = {id(k): v for k, v in other.items()}
        for k, v in self.items():
            ov = other_map.get(id(k), Ellipsis)
            if ov is Ellipsis or ov != v:
                return False
        return True


In [35]:
type(dict.items)

method_descriptor